# PP4AV EDA: what the training design depends on

PP4AV: 3,447 street images from 6 European cities (+ fisheye), with **face** and **license plate** bounding boxes.
Training will use each box filled in as a mask.

This notebook measures three properties of the data. Each one decides one training choice:

| # | measurement | why it matters | decides |
|---|---|---|---|
| 1 | object size in pixels vs. 16 px | Segmenter labels the image in 16×16 patches; an object much smaller than a patch can barely be represented | full resolution vs. upscaling |
| 2 | fraction of pixels that are face / plate | if tiny, "background everywhere" already looks near-perfect to the loss | class weighting / object-aware crop sampling |
| 3 | similarity of images within each city | frames come from driving videos; near-duplicates in both train and held-out images make inference examples look better than the model is | random split vs. holding out whole cities |

In [ ]:
import zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

DATA_ROOT = Path("/data/pp4av")   # = ~/mita/data/pp4av on the host
HF_REPO = "khaclinh/pp4av"
PATCH = 16                        # ViT patch size: the model decides labels per 16x16 block
CROP = 512                        # MiTA-Segmenter training crop size
CLASS_NAMES = {0: "face", 1: "plate"}
CLASS_COLORS = {"face": "red", "plate": "cyan"}

## 0. Download

Downloads the two zips from Hugging Face once (~4.5 GB) and extracts them. Re-running skips what is already there.

In [ ]:
def fetch_and_extract(zip_name, out_name):
    out_dir = DATA_ROOT / out_name
    if out_dir.exists() and any(out_dir.iterdir()):
        print(f"already extracted: {out_dir}")
        return out_dir
    zip_path = DATA_ROOT / "raw" / "data" / zip_name
    if not zip_path.exists():
        from huggingface_hub import hf_hub_download
        hf_hub_download(HF_REPO, f"data/{zip_name}", repo_type="dataset", local_dir=DATA_ROOT / "raw")
    print(f"extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)
    return out_dir

IMG_DIR = fetch_and_extract("images.zip", "images")
ANN_DIR = fetch_and_extract("annotations.zip", "annotations")

## 1. Index: one table of images, one table of boxes

Each label file holds one line per object: `class x_center y_center width height`, all as fractions of the image size
(class 0 = face, 1 = plate). Converted here to pixel corners. The subset (city) is the image's folder name.

In [ ]:
IMG_EXTS = {".png", ".jpg", ".jpeg"}
image_paths = sorted(p for p in IMG_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS)
label_files = {(p.parent.name, p.stem): p for p in ANN_DIR.rglob("*.txt")}

img_rows, box_rows = [], []
for p in image_paths:
    subset = p.parent.name
    with Image.open(p) as im:            # reads the header only, not the pixels
        W, H = im.size
    ann = label_files.get((subset, p.stem))
    img_rows.append(dict(image=str(p), subset=subset, W=W, H=H, has_label_file=ann is not None))
    if ann is None:
        continue
    for line in ann.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        c, cx, cy, w, h = int(parts[0]), *map(float, parts[1:5])
        box_rows.append(dict(image=str(p), subset=subset, cls=CLASS_NAMES.get(c, f"class{c}"),
                             x1=(cx - w / 2) * W, y1=(cy - h / 2) * H,
                             x2=(cx + w / 2) * W, y2=(cy + h / 2) * H))

imgs = pd.DataFrame(img_rows)
boxes = pd.DataFrame(box_rows, columns=["image", "subset", "cls", "x1", "y1", "x2", "y2"])
boxes["w"] = boxes.x2 - boxes.x1
boxes["h"] = boxes.y2 - boxes.y1
boxes["min_side"] = np.minimum(boxes.w, boxes.h)
print(f"{len(imgs)} images, {len(boxes)} boxes")

## 2. Sanity: counts, resolutions, missing labels

In [ ]:
n_obj = boxes.groupby("image").size()
imgs["n_objects"] = imgs.image.map(n_obj).fillna(0).astype(int)

summary = imgs.groupby("subset").agg(
    images=("image", "size"),
    resolution=("W", lambda s: ", ".join(sorted({f"{w}x{h}" for w, h in zip(s, imgs.loc[s.index, "H"])}))),
    no_label_file=("has_label_file", lambda s: int((~s).sum())),
    no_objects=("n_objects", lambda s: int((s == 0).sum())),
)
summary = summary.join(pd.crosstab(boxes.subset, boxes.cls)).fillna(0)
summary.loc["TOTAL"] = summary.sum(numeric_only=True)
summary.loc["TOTAL", "resolution"] = ""
summary

### Samples with boxes

One image per subset (red = face, cyan = plate), then enlarged crops around individual objects, titled with their size in pixels.

In [ ]:
def draw_boxes(ax, image_path):
    ax.imshow(Image.open(image_path).convert("RGB"))
    for b in boxes[boxes.image == image_path].itertuples():
        ax.add_patch(patches.Rectangle((b.x1, b.y1), b.w, b.h, fill=False, lw=2,
                                       edgecolor=CLASS_COLORS.get(b.cls, "yellow")))
    ax.axis("off")

rng = np.random.default_rng(0)
subsets = sorted(imgs.subset.unique())
cols = 2
fig, axes = plt.subplots(int(np.ceil(len(subsets) / cols)), cols, figsize=(16, 4.5 * np.ceil(len(subsets) / cols)), squeeze=False)
for ax, s in zip(axes.flat, subsets):
    cand = imgs[(imgs.subset == s) & (imgs.n_objects > 0)]
    if len(cand) == 0:
        cand = imgs[imgs.subset == s]
    row = cand.iloc[rng.integers(len(cand))]
    draw_boxes(ax, row.image)
    ax.set_title(f"{s}  ({row.n_objects} objects)")
for ax in axes.flat[len(subsets):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
def show_object_crops(cls, n=16, pad=2.0):
    sel = boxes[boxes.cls == cls]
    sel = sel.sample(min(n, len(sel)), random_state=0)
    fig, axes = plt.subplots(2, int(np.ceil(len(sel) / 2)), figsize=(18, 5), squeeze=False)
    for ax, b in zip(axes.flat, sel.itertuples()):
        im = Image.open(b.image).convert("RGB")
        cx, cy = (b.x1 + b.x2) / 2, (b.y1 + b.y2) / 2
        r = max(b.w, b.h) * pad / 2 + 4
        ax.imshow(im.crop((int(cx - r), int(cy - r), int(cx + r), int(cy + r))), interpolation="nearest")
        ax.set_title(f"{b.w:.0f}x{b.h:.0f}px", fontsize=9)
        ax.axis("off")
    for ax in axes.flat[len(sel):]:
        ax.axis("off")
    fig.suptitle(f"{cls}: random objects, enlarged (title = box size in original pixels)")
    plt.tight_layout(); plt.show()

for cls in boxes.cls.unique():
    show_object_crops(cls)

## 3. Measurement 1: object size vs. the 16 px patch

The model sees the image as a grid of 16×16 patches and scores each patch per class; the score maps are then upsampled.
An object whose smaller side is under 16 px covers less than one patch, so it has to win a patch it only partly occupies.

**Read off:** the share of objects under 16 px. If it is large, training at native resolution is not enough and images should be upscaled before cropping.

In [ ]:
bins = np.logspace(0, np.log10(max(boxes.min_side.max(), 64) * 1.1), 40)
fig, ax = plt.subplots(figsize=(10, 4))
for cls, g in boxes.groupby("cls"):
    ax.hist(g.min_side.clip(lower=1), bins=bins, alpha=0.6, label=f"{cls} (n={len(g)})",
            color=CLASS_COLORS.get(cls))
for v, label in [(PATCH, "1 patch (16px)"), (2 * PATCH, "2 patches")]:
    ax.axvline(v, color="k", ls="--", lw=1)
    ax.text(v, ax.get_ylim()[1] * 0.95, " " + label, va="top")
ax.set_xscale("log"); ax.set_xlabel("smaller box side (px, original resolution)"); ax.set_ylabel("objects")
ax.legend(); plt.show()

def size_table(df):
    return pd.Series({
        "objects": len(df),
        "median px": df.min_side.median(),
        "% < 8px": 100 * (df.min_side < 8).mean(),
        "% < 16px": 100 * (df.min_side < PATCH).mean(),
        "% < 32px": 100 * (df.min_side < 2 * PATCH).mean(),
    })

size_by_class = boxes.groupby("cls")[["min_side"]].apply(size_table).round(1)
size_by_subset = boxes.groupby(["cls", "subset"])[["min_side"]].apply(size_table).round(1)
display(size_by_class)
display(size_by_subset)

## 4. Measurement 2: how rare face / plate pixels are

Two views of the same imbalance:

- **pixel share**: the fraction of all pixels inside a face or plate box. This is what an unweighted per-pixel loss sees.
- **crop hit rate**: training sees random 512×512 crops, not whole images. The share of random crops containing at least one object is the share of training steps that carry any signal for the rare classes.

**Read off:** if the pixel share is far below 1% or the hit rate is low, the loss needs class weights and/or crops centered on objects.

In [ ]:
def pixel_share(scale=4):
    """Rasterize boxes at 1/scale resolution; overlapping boxes count once."""
    totals = {"face": 0, "plate": 0, "background": 0}
    per_image = []
    by_img = dict(tuple(boxes.groupby("image")))
    for r in imgs.itertuples():
        h, w = r.H // scale, r.W // scale
        lab = np.zeros((h, w), np.uint8)              # 0 background, 1 face, 2 plate
        for b in by_img.get(r.image, pd.DataFrame()).itertuples():
            v = 1 if b.cls == "face" else 2
            lab[max(0, int(b.y1 // scale)):int(np.ceil(b.y2 / scale)),
                max(0, int(b.x1 // scale)):int(np.ceil(b.x2 / scale))] = v
        n = lab.size
        f, p = int((lab == 1).sum()), int((lab == 2).sum())
        totals["face"] += f; totals["plate"] += p; totals["background"] += n - f - p
        per_image.append(100 * (f + p) / n)
    all_px = sum(totals.values())
    return {k: 100 * v / all_px for k, v in totals.items()}, np.array(per_image)

share, per_image_fg = pixel_share()
print("pixel share over the whole dataset (%):", {k: round(v, 4) for k, v in share.items()})

def crop_hit_rate(n_crops_per_image=20, seed=0):
    rng = np.random.default_rng(seed)
    cx = ((boxes.x1 + boxes.x2) / 2).to_numpy(); cy = ((boxes.y1 + boxes.y2) / 2).to_numpy()
    idx_by_img = boxes.groupby("image").indices
    hits = []
    for r in imgs.itertuples():
        ii = idx_by_img.get(r.image, np.array([], int))
        cw, ch = min(CROP, r.W), min(CROP, r.H)
        x0 = rng.integers(0, r.W - cw + 1, n_crops_per_image)
        y0 = rng.integers(0, r.H - ch + 1, n_crops_per_image)
        if len(ii) == 0:
            hits.extend([False] * n_crops_per_image); continue
        inside = ((cx[ii][None] >= x0[:, None]) & (cx[ii][None] < x0[:, None] + cw) &
                  (cy[ii][None] >= y0[:, None]) & (cy[ii][None] < y0[:, None] + ch))
        hits.extend(inside.any(1))
    return 100 * np.mean(hits)

hit_rate = crop_hit_rate()
print(f"random {CROP}x{CROP} crops containing >= 1 object: {hit_rate:.1f}%")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.hist(per_image_fg, bins=50)
ax.set_xlabel("% of the image covered by face + plate boxes"); ax.set_ylabel("images")
plt.show()

## 5. Measurement 3: how similar images are within a city

Each subset is frames sampled from one driving video, so neighbouring frames may be almost identical.
Every image is shrunk to a 64×36 grayscale thumbnail (normalized for brightness/contrast) and compared to all others:

- **nearest neighbour in the same subset**: how close its closest sibling frame is
- **random pair in the same subset**: what "same city, different moment" looks like
- **nearest neighbour in other subsets**: what "different video" looks like

**Read off:** if same-subset nearest neighbours sit far below random same-subset pairs, near-duplicates exist, and held-out
images for the inference examples should be whole subsets, not random images. The pairs shown below are the direct evidence.

In [ ]:
TW, TH = 64, 36
thumb_cache = DATA_ROOT / "eda_thumbs.npy"

def thumb(path):
    with Image.open(path) as im:
        t = np.asarray(im.convert("L").resize((TW, TH), Image.BILINEAR), np.float32).ravel()
    return (t - t.mean()) / (t.std() + 1e-6)

if thumb_cache.exists() and len(np.load(thumb_cache)) == len(imgs):
    T = np.load(thumb_cache)
else:
    with ThreadPoolExecutor(16) as ex:
        T = np.stack(list(ex.map(thumb, imgs.image)))
    np.save(thumb_cache, T)

# distance per pixel between normalized thumbnails (0 = identical)
sq = (T ** 2).sum(1)
D = np.sqrt(np.maximum(sq[:, None] + sq[None] - 2 * T @ T.T, 0) / T.shape[1])
np.fill_diagonal(D, np.inf)
sub = imgs.subset.to_numpy()
same = sub[:, None] == sub[None]

nn_same = np.where(same, D, np.inf).min(1)
nn_other = np.where(~same, D, np.inf).min(1)
rng = np.random.default_rng(0)
rand_same = []
for i in range(len(imgs)):
    js = np.flatnonzero(same[i] & np.isfinite(D[i]))
    if len(js):
        rand_same.append(D[i, rng.choice(js)])

fig, ax = plt.subplots(figsize=(10, 4))
for vals, label in [(nn_same[np.isfinite(nn_same)], "nearest neighbour, same subset"),
                    (np.array(rand_same), "random pair, same subset"),
                    (nn_other[np.isfinite(nn_other)], "nearest neighbour, other subsets")]:
    ax.hist(vals, bins=50, alpha=0.55, label=label)
ax.set_xlabel("thumbnail distance (0 = identical)"); ax.set_ylabel("images"); ax.legend()
plt.show()

dup = pd.DataFrame({"subset": sub, "nn_same": nn_same, "nn_other": nn_other})
display(dup.groupby("subset").agg(images=("nn_same", "size"),
                                  median_nn_same=("nn_same", "median"),
                                  median_nn_other=("nn_other", "median")).round(3))

In [ ]:
# the 6 closest same-subset pairs, side by side
order = np.argsort(np.where(same, D, np.inf), axis=None)
shown, pairs = set(), []
for flat in order:
    i, j = divmod(int(flat), len(imgs))
    if not np.isfinite(D[i, j]):
        break
    if i in shown or j in shown:
        continue
    pairs.append((i, j)); shown |= {i, j}
    if len(pairs) == 6:
        break

fig, axes = plt.subplots(len(pairs), 2, figsize=(14, 3.3 * len(pairs)), squeeze=False)
for row, (i, j) in zip(axes, pairs):
    for ax, k in zip(row, (i, j)):
        ax.imshow(Image.open(imgs.image[k]).convert("RGB")); ax.axis("off")
        ax.set_title(f"{imgs.subset[k]} / {Path(imgs.image[k]).name}", fontsize=9)
    row[0].text(1.02, 0.5, f"d={D[i, j]:.3f}", transform=row[0].transAxes, va="center")
plt.tight_layout(); plt.show()

## 6. Summary: the three numbers

In [ ]:
summary_rows = []
for cls in sorted(boxes.cls.unique()):
    summary_rows.append(("1. size", f"{cls}: objects under 16 px", f"{size_by_class.loc[cls, '% < 16px']:.1f}%"))
summary_rows += [
    ("2. imbalance", "face pixel share", f"{share['face']:.3f}%"),
    ("2. imbalance", "plate pixel share", f"{share['plate']:.3f}%"),
    ("2. imbalance", f"random {CROP}px crops with >= 1 object", f"{hit_rate:.1f}%"),
    ("3. duplicates", "median NN distance, same subset", f"{np.median(nn_same[np.isfinite(nn_same)]):.3f}"),
    ("3. duplicates", "median distance, random same-subset pair", f"{np.median(rand_same):.3f}"),
    ("3. duplicates", "median NN distance, other subsets", f"{np.median(nn_other[np.isfinite(nn_other)]):.3f}"),
]
pd.DataFrame(summary_rows, columns=["measurement", "quantity", "value"]).set_index(["measurement", "quantity"])